# Notebook 09 — All Run: Pipeline Đầy Đủ Tuần 4
**Nhóm 67 | Tuần 4 | Ngôn ngữ Lập trình Python**

Notebook này chạy **toàn bộ pipeline** từ đầu đến cuối theo đúng thứ tự, gom lại từ tất cả các notebook riêng lẻ (00 → 08).

**Thứ tự thực hiện:**
1. Thiết lập môi trường & kiểm tra file
2. Đồng bộ dữ liệu (regenerate)
3. Mô phỏng 100 bài nộp sinh viên
4. EDA: Phân tích bộ dữ liệu MBPP_100
5. Kiểm thử Secure Sandbox v3
6. Chạy Baseline (kiểm chứng solution chuẩn)
7. Chấm bài tự động 100 bài nộp
8. So sánh 4 cấu hình chấm (RQ1)
9. Trực quan hóa kết quả (6 biểu đồ)
10. Khởi chạy Web Demo Streamlit


---
## PHẦN A — Thiết lập Môi trường
*Từ Notebook 00 — setup_v3*

### A1 — Kết nối Google Drive & Thiết lập BASE path

In [ ]:
import os
import sys
import json
import subprocess
import threading
import time
from pathlib import Path

# ── Kết nối Google Drive (chỉ cần trên Colab) ──
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/Project')
    print('✓ Đã kết nối Google Drive (Colab mode)')
except ImportError:
    # Local machine: tự detect từ vị trí notebook
    cwd = Path.cwd()
    if cwd.name in {'notebookes', 'notebooks', 'src', 'data', 'results'}:
        BASE = cwd.parent
    else:
        BASE = cwd
    print(f'✓ Local mode — BASE = {BASE}')

sys.path.insert(0, str(BASE / 'src'))
os.chdir(BASE)
print(f'✓ BASE = {BASE}')
print(f'✓ Working dir: {os.getcwd()}')


### A2 — Cài đặt thư viện (Colab)

In [ ]:
# Cài thư viện cần thiết (bỏ qua nếu đã cài)
import subprocess, sys

libs = [
    'psutil', 'pandas', 'matplotlib', 'seaborn',
    'streamlit', 'streamlit-ace', 'autopep8'
]
print('Kiểm tra & cài thư viện...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + libs, check=False)
print('✅ Thư viện đã sẵn sàng')


### A3 — Tạo cấu trúc thư mục

In [ ]:
dirs = ['data/raw', 'data/processed', 'notebookes', 'src', 'results', 'report']
for d in dirs:
    (BASE / d).mkdir(parents=True, exist_ok=True)
print('✅ Đã tạo cấu trúc thư mục')


### A4 — Kiểm tra file cần thiết

In [ ]:
files_to_check = {
    'runner_v3.py'       : BASE / 'src' / 'runner_v3.py',
    'hidden_v2.json'     : BASE / 'data' / 'processed' / 'hidden_v2.json',
    'submissions_50.json': BASE / 'data' / 'processed' / 'submissions_50.json',
    'feedback.py'        : BASE / 'src' / 'feedback.py',
    'comparison_3sets.py': BASE / 'src' / 'comparison_3sets.py',
    'run_grading_v2.py'  : BASE / 'src' / 'run_grading_v2.py',
    'run_baseline.py'    : BASE / 'src' / 'run_baseline.py',
    'generate_plots.py'  : BASE / 'src' / 'generate_plots.py',
    'app.py'             : BASE / 'app.py',
}
all_ok = True
for name, path in files_to_check.items():
    icon = '✓' if path.exists() else '✗ THIẾU'
    print(f'  [{icon}] {name}')
    if not path.exists(): all_ok = False

print()
if all_ok:
    print('✅ Đủ file — môi trường sẵn sàng chạy pipeline')
else:
    print('❌ Thiếu file — kiểm tra lại thư mục src/ và data/processed/')


### A5 — Kiểm tra import runner_v3

In [ ]:
try:
    from runner_v3 import grade_submission, compute_leakage
    from feedback import generate_vietnamese_feedback
    print('✓ runner_v3.py import thành công')
    print('✓ feedback.py import thành công')
    print('\n✅ Môi trường sẵn sàng — tiếp tục Phần B')
except Exception as e:
    print(f'✗ Lỗi import: {e}')
    print('→ Kiểm tra lại sys.path và thư mục src/')


---
## PHẦN B — Đồng bộ Dữ liệu
*Từ regenerate_all.py — tạo hidden_v2.json, mbpp_clean.json, submissions_50.json*

### B1 — Kiểm tra & chạy regenerate_all.py

In [ ]:
hidden_file = BASE / 'data' / 'processed' / 'hidden_v2.json'
subs_file   = BASE / 'data' / 'processed' / 'submissions_50.json'

if hidden_file.exists() and subs_file.exists():
    print('[SKIP] Dữ liệu đã tồn tại — bỏ qua regenerate_all.py')
    print(f'  ✓ {hidden_file.name}')
    print(f'  ✓ {subs_file.name}')
    print('  → Xóa 2 file trên nếu muốn tái tạo dữ liệu từ đầu')
else:
    print('[RUN] Đang chạy regenerate_all.py...')
    import regenerate_all
    regenerate_all.main()
    print('\n✅ Đồng bộ dữ liệu hoàn tất')


---
## PHẦN C — Mô phỏng Bài nộp Sinh viên
*Từ Notebook 03 — simulate_v3*

### C1 — Chạy simulate_v2.generate() và kiểm tra submissions_50.json

In [ ]:
import simulate_v2
print('✓ Bắt đầu đồng bộ dữ liệu bài nộp sinh viên (100 bài)...\n')
simulate_v2.generate()


### C2 — Xem mẫu 3 bài nộp đầu tiên

In [ ]:
import json
SUB_FILE = BASE / 'data' / 'processed' / 'submissions_50.json'
with open(SUB_FILE, encoding='utf-8') as f:
    subs = json.load(f)

print(f'✓ Đã load {len(subs)} bài nộp mô phỏng\n')

from collections import Counter
print('Phân bố error_type:')
for k, v in sorted(Counter(s['error_type'] for s in subs).items()):
    print(f'  {k:20s}: {v} bài')
print()
print('Phân bố topic:')
for k, v in sorted(Counter(s['topic'] for s in subs).items()):
    print(f'  {k:10s}: {v} bài')

print('\nMẫu 3 bài nộp đầu tiên:')
for sub in subs[:3]:
    print(f'  [{sub["submission_id"]}] Task {sub["task_id"]} | {sub["func_name"]}() | {sub["error_type"]} | {sub["note"][:60]}')


---
## PHẦN D — EDA: Phân tích Bộ dữ liệu MBPP_100
*Từ Notebook 01 — eda_v3*

### D1 — Load dữ liệu và thống kê mô tả

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_FILE = BASE / 'data' / 'processed' / 'hidden_v2.json'
with open(DATA_FILE, encoding='utf-8') as f:
    problems = json.load(f)

df = pd.DataFrame(problems)
df['num_public_tests'] = df['public_tests'].apply(len)
df['num_hidden_tests'] = df['hidden_tests'].apply(len)
df['desc_word_count']  = df['text'].apply(lambda x: len(x.split()))
df['code_line_count']  = df['code'].apply(lambda x: len(x.strip().split('\n')))

print('=============================================================')
print('  BẢNG THỐNG KÊ MÔ TẢ BỘ DỮ LIỆU MBPP_100')
print('=============================================================')
print(f'  Tổng số bài toán                   : {len(df)}')
print(f'  Phân bố chủ đề:')
for topic, count in df['topic'].value_counts().items():
    print(f'    - {topic:<15}: {count} bài')
print(f'  Public tests trung bình/bài         : {df["num_public_tests"].mean():.1f} (min={df["num_public_tests"].min()}, max={df["num_public_tests"].max()})')
print(f'  Hidden tests trung bình/bài         : {df["num_hidden_tests"].mean():.1f} (min={df["num_hidden_tests"].min()}, max={df["num_hidden_tests"].max()})')
print(f'  Độ dài mô tả trung bình (từ)        : {df["desc_word_count"].mean():.1f}')
print(f'  Độ dài code mẫu trung bình (dòng)   : {df["code_line_count"].mean():.1f}')
print('=============================================================')


### D2 — Biểu đồ phân bố chủ đề và số hidden tests

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=120)

# Plot 1: Topic distribution
sns.countplot(data=df, x='topic', palette='viridis', ax=axes[0])
axes[0].set_title('Topic Distribution — MBPP_100', fontweight='bold')
axes[0].set_xlabel('Chủ đề'); axes[0].set_ylabel('Số bài toán')

# Plot 2: Hidden test count distribution
df['num_hidden_tests'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='#5B9BD5', edgecolor='white')
axes[1].set_title('Số Hidden Tests mỗi bài (Bell Curve)', fontweight='bold')
axes[1].set_xlabel('Số hidden tests'); axes[1].set_ylabel('Số bài toán')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(BASE / 'results' / 'topic_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Đã lưu: results/topic_distribution.png')


---
## PHẦN E — Kiểm thử Secure Sandbox v3
*Từ Notebook 02 — runner_v3 | Stress-test 13 kịch bản tấn công*

### E1 — Import Sandbox v3 và feedback module

In [ ]:
from runner_v3 import grade_submission as grade_v3
from feedback import generate_vietnamese_feedback
print('✓ Import Secure Sandbox v3 thành công!')


### E2 — Kiểm tra bộ lọc AST (import cấm, eval, dunder)

In [ ]:
test_cases_ast = [
    ('import os\ndef hack(x):\n    return os.getcwd()', 'hack', 'Import cấm os'),
    ('def eval_hack(x):\n    eval("print(1)")\n    return x',   'eval_hack', 'Gọi eval()'),
    ('def dunder(x):\n    return x.__class__.__base__',           'dunder',    'Truy cập __class__'),
]
dummy_tests = [{'input': '1', 'expected': '1'}]

print('--- Kiểm tra bộ lọc AST Whitelist ---')
for code_str, func, label in test_cases_ast:
    r = grade_v3(code_str, func, dummy_tests)
    status = r['test_results'][0]['status']
    icon = '✅ CHẶN' if status == 'SE' else f'❌ BỎ SÓT [{status}]'
    print(f'  {icon} | {label}')


### E3 — Kiểm tra giới hạn tài nguyên (TLE & MLE)

In [ ]:
test_cases_resource = [
    ('def infinite_loop(x):\n    while True:\n        pass',                             'infinite_loop', 'TLE — vòng lặp vô hạn'),
    ('def memory_leak(x):\n    y = " " * (200 * 1024 * 1024)\n    return len(y)',       'memory_leak',   'MLE — cấp phát 200MB'),
    ('def recurse(x):\n    return recurse(x + 1)',                                        'recurse',       'RecursionError — đệ quy vô hạn'),
]
dummy_tests = [{'input': '1', 'expected': '1'}]

print('--- Kiểm tra giới hạn tài nguyên ---')
for code_str, func, label in test_cases_resource:
    r = grade_v3(code_str, func, dummy_tests)
    status = r['test_results'][0]['status']
    print(f'  [{status}] {label}')


### E4 — Kiểm tra phản hồi sư phạm tiếng Việt

In [ ]:
zero_div_code = 'def average(lst):\n    return sum(lst) / len(lst)'
tests = [{'input': '[]', 'expected': 'None'}]
r_div = grade_v3(zero_div_code, 'average', tests)
test_res = r_div['test_results'][0]

print('--- Phản hồi sư phạm tiếng Việt ---')
print(f'Lớp lỗi nhận diện : {test_res["status"]}')
print(f'Traceback gốc     :\n{test_res["error_msg"][:200]}')
print()
fb = generate_vietnamese_feedback(r_div)
print(fb)


---
## PHẦN F — Baseline Evaluation
*Từ Notebook 06 — kiểm chứng solution chuẩn, mọi bài phải PASS 100%*

### F1 — Chạy baseline song song (ThreadPoolExecutor)

In [ ]:
import run_baseline
print('Đang chạy baseline trên toàn bộ bài toán...\n')
run_baseline.main()


### F2 — Load & kiểm tra kết quả baseline

In [ ]:
df_bl = pd.read_csv(BASE / 'results' / 'baseline_summary.csv')

n_pass = df_bl['baseline_ok'].sum()
n_total = len(df_bl)
print(f'Baseline PASS: {n_pass}/{n_total} bài')
if n_pass < n_total:
    print('⚠️  Có bài FAIL — kiểm tra test case!')
    fail_cols = ['task_id', 'func_name', 'topic', 'pub_tpr', 'hid_tpr']
    display(df_bl[df_bl['baseline_ok'] == False][fail_cols])
else:
    print('✅ Tất cả bài PASS — bộ test case hợp lệ!')

print(f'\nLatency public trung bình : {df_bl["pub_latency"].mean():.4f}s/test')
print(f'Latency hidden trung bình  : {df_bl["hid_latency"].mean():.4f}s/test')
print(f'Latency tổng trung bình    : {df_bl["avg_latency_total"].mean():.4f}s/test')


### F3 — Phân tích latency theo chủ đề

In [ ]:
lat_by_topic = df_bl.groupby('topic')['avg_latency_total'].agg(['mean','min','max','std']).round(4)
lat_by_topic.columns = ['Trung bình (s)', 'Min (s)', 'Max (s)', 'Std (s)']
print('Latency theo chủ đề bài toán:')
display(lat_by_topic)


---
## PHẦN G — Chấm Bài Tự Động 100 Bài Nộp
*Từ run_grading_v2.py — dùng runner_v3, sinh error_analysis_v2.csv*

### G1 — Chạy chấm bài (runner_v3, 3 public + 6-10 hidden)

In [ ]:
import run_grading_v2
print('Đang chấm 100 bài nộp mô phỏng...\n')
run_grading_v2.main()


### G2 — Tổng hợp kết quả chấm bài

In [ ]:
df_grading = pd.read_csv(BASE / 'results' / 'error_analysis_v2.csv')
print(f'Tổng bài chấm: {len(df_grading)}')
print(f'\nPhân bố trạng thái chấm bài:')
if 'is_false_positive' in df_grading.columns:
    fp = df_grading['is_false_positive'].sum()
    print(f'  False Positive (pub=100%, hid<100%): {fp} bài ({fp/len(df_grading)*100:.1f}%)')
print(f'\nLatency trung bình (hidden) : {df_grading["avg_latency_hid"].mean():.4f}s/test')
print(f'Latency trung bình (public) : {df_grading["avg_latency_pub"].mean():.4f}s/test')
display(df_grading[['sv_id','task_id','func','topic','pub_tpr','hid_tpr','is_false_positive']].head(10))


---
## PHẦN H — So sánh 4 Cấu hình Chấm bài (RQ1)
*Từ Notebook 05 — comparison_v3 | Sinh comparison_week4.csv*

### H1 — Chạy comparison_3sets.main() (4 Set)

In [ ]:
import comparison_3sets
print('Đang chạy so sánh 4 cấu hình kiểm thử...\n')
comparison_3sets.main()


### H2 — Hiển thị bảng kết quả comparison_week4.csv

In [ ]:
df_comp = pd.read_csv(BASE / 'results' / 'comparison_week4.csv')
print(f'Tổng: {len(df_comp)} bài nộp được so sánh')
print(f'Columns: {list(df_comp.columns)[:8]}...')

cols_show = [c for c in ['sv_id','task_id','func','topic','actual_error_type',
                           'set1_tpr','set2_tpr','set3_tpr','set4_tpr',
                           'is_fp_set1','is_fp_set2','is_fp_set3','is_fp_set4'] if c in df_comp.columns]
display(df_comp[cols_show].head(10))


---
## PHẦN I — Trực quan hóa Kết quả (6 Biểu đồ)
*Từ Notebook 04 — visualize_v3 | Đọc từ comparison_week4.csv*

### I1 — Kiểm tra file đầu vào và chạy generate_plots.main()

In [ ]:
import generate_plots

csv_file = BASE / 'results' / 'comparison_week4.csv'
if not csv_file.exists():
    print(f'[LỖI] Chưa có comparison_week4.csv — hãy chạy Phần H trước!')
else:
    print(f'[OK] Tìm thấy comparison_week4.csv — bắt đầu vẽ biểu đồ...')
    generate_plots.main()


### I2 — Hiển thị toàn bộ 6 biểu đồ kết quả

In [ ]:
from IPython.display import Image, display as ipy_display

plots = [
    ('1. Đường cong FPR theo số test case (RQ1)',    'FPR_vs_ntest.png'),
    ('2. Phân bố lỗi SE/WA/RE/TLE/MLE (4 bộ test)', 'error_types_comparison.png'),
    ('3. FPR theo chủ đề bài toán',                  'fpr_by_topic.png'),
    ('4. So sánh Latency (4 Set)',                    'latency_comparison.png'),
    ('5. Full-run vs Fail-Fast Latency',              'full_vs_failfast_latency.png'),
    ('6. Phân bố độ dài mô tả & số test/bài',        'description_lengths_distribution.png'),
]

for title, fname in plots:
    fpath = BASE / 'results' / fname
    if fpath.exists():
        print(f'\n{title}:')
        ipy_display(Image(filename=str(fpath)))
    else:
        print(f'[THIẾU] {fname} — chưa được tạo')


---
## PHẦN J — Khởi chạy Web Demo Streamlit
*Từ Notebook 08 — streamlit_app | URL: http://localhost:8501*

### J1 — Kiểm tra app.py và khởi động Streamlit

In [ ]:
app_path = BASE / 'app.py'
print(f'✓ BASE = {BASE}')
print(f'✓ app.py exists: {app_path.exists()}')
if not app_path.exists():
    raise FileNotFoundError(f'Không tìm thấy {app_path}')

# ── Local: chạy trực tiếp ──────────────────────────────────────
# Bỏ comment 3 dòng dưới nếu chạy trên Local Machine:
# streamlit_proc = subprocess.Popen(
#     [sys.executable, '-m', 'streamlit', 'run', str(app_path), '--server.port', '8501'],
#     cwd=str(BASE))
# print(f'Streamlit PID: {streamlit_proc.pid} | URL: http://localhost:8501')

# ── Colab: dùng pyngrok ────────────────────────────────────────
try:
    from pyngrok import ngrok
    def _run_st():
        subprocess.run([sys.executable, '-m', 'streamlit', 'run', str(app_path),
                        '--server.port', '8501', '--server.headless', 'true'], cwd=str(BASE))
    threading.Thread(target=_run_st, daemon=True).start()
    time.sleep(6)
    public_url = ngrok.connect(8501)
    print(f'✅ Web Demo đang chạy tại: {public_url}')
    print('   (Colab mode — dùng pyngrok)')
except ImportError:
    print('pyngrok chưa được cài — chạy: !pip install pyngrok')
    print('Hoặc bỏ comment phần Local ở trên nếu chạy trên máy cá nhân')


### J2 — Dừng Streamlit (chạy khi không cần nữa)

In [ ]:
# Dừng ngrok tunnel (Colab)
try:
    from pyngrok import ngrok as _ngrok
    _ngrok.kill()
    print('✓ Đã dừng ngrok tunnel')
except Exception:
    pass

# Dừng process (Local)
if 'streamlit_proc' in globals() and streamlit_proc.poll() is None:
    streamlit_proc.terminate()
    streamlit_proc.wait(timeout=10)
    print('✓ Đã dừng Streamlit')
else:
    print('Streamlit chưa chạy hoặc đã dừng')


---
## PHẦN K — Tổng kết Pipeline

### K1 — Kiểm tra toàn bộ file kết quả đầu ra

In [ ]:
print('=' * 60)
print('  TỔNG KẾT FILE KẾT QUẢ ĐẦU RA')
print('=' * 60)

output_files = {
    'Baseline'          : BASE / 'results' / 'baseline_summary.csv',
    'Baseline JSON'     : BASE / 'results' / 'baseline_summary.json',
    'Grading CSV'       : BASE / 'results' / 'error_analysis_v2.csv',
    'Grading JSON'      : BASE / 'results' / 'error_analysis_v2.json',
    'Comparison CSV'    : BASE / 'results' / 'comparison_week4.csv',
    'Comparison JSON'   : BASE / 'results' / 'comparison_week4.json',
    'FPR chart'         : BASE / 'results' / 'FPR_vs_ntest.png',
    'Error chart'       : BASE / 'results' / 'error_types_comparison.png',
    'FPR by topic'      : BASE / 'results' / 'fpr_by_topic.png',
    'Latency chart'     : BASE / 'results' / 'latency_comparison.png',
    'Fail-fast chart'   : BASE / 'results' / 'full_vs_failfast_latency.png',
    'Desc lengths chart': BASE / 'results' / 'description_lengths_distribution.png',
    'Topic chart'       : BASE / 'results' / 'topic_distribution.png',
}

n_ok = 0
for name, path in output_files.items():
    icon = '✓' if path.exists() else '✗ THIẾU'
    size = f'{path.stat().st_size/1024:.1f} KB' if path.exists() else ''
    print(f'  [{icon}] {name:<22} {size}')
    if path.exists(): n_ok += 1

print()
print(f'  Kết quả: {n_ok}/{len(output_files)} file đã được tạo')
if n_ok == len(output_files):
    print('  ✅ Pipeline hoàn chỉnh!')
else:
    print('  ⚠️  Còn thiếu file — kiểm tra lại các bước bị lỗi ở trên')
print('=' * 60)
